# Laboratorio: Búsqueda en entornos complejos

## Búsqueda Local 

En este laboratorio se estudiarán algoritmos de búsqueda local y heurística en un entorno de espacio de estados complejo. El problema elegido es el de las 8 reinas, en el cual se deben ubicar 8 reinas sobre un tablero de ajedrez de 8x8 de manera que ninguna reina pueda atacar a otra. 

A diferencia de los métodos de búsqueda sistemática, los algoritmos de búsqueda local no construyen explícitamente un árbol completo, sino que exploran el espacio de estados moviéndose entre configuraciones vecinas. 

## Descripción del problema

El problema de las 8 reinas consiste en colocar 8 reinas sobre un tablero de 8x8 de forma que:
- no compartan la misma fila
- no compartan la misma columna
- no compartan la misma diagonal.

*Representación del estado*

Cada estado se representará como un arreglo de longitud 8:



In [1]:
estado = [0, 4, 7, 5, 2, 6, 1, 3]

significa:
- en la columna 0 hay una reina en la fila 0,
- en la columna 1 hay una reina en la fila 4,
- ...
- en la columna 7 hay una reina en la fila 3.

## Implementación

*En parejas*, deberán implementar: 
- Hill Climbing
- Una variación del Hill Climbing (Stochastic/ First-choice/ Random-restart)
- Beam Search con beam width = _k_

Cada grupo debe realizar experimentos comparativos entre los algoritmos implementados.

### Experimentos requeridos:

Ejecutar cada algoritmo 1000 veces con estados iniciales aleatorios y reportar:

- porcentaje de éxito,
- promedio de largo de episodio (hasta éxito o estancarse)
- promedio de valor heurístico final,
- tiempo promedio de ejecución.

Para Beam Search, repetir los experimentos con distintos valores de _k_ (2,5,10). 

## Discuta 

¿Qué tan frecuentemente Hill Climbing encuentra una solución?

¿Qué tipo de problemas presenta Hill Climbing en este dominio?

¿La variante elegida mejora el desempeño? ¿Por qué?

¿Cómo afecta el valor de k en Beam Search?

¿Cuál algoritmo resultó más efectivo?

¿Qué relación existe entre costo computacional y tasa de éxito?

## Funciones útiles

In [2]:
def es_solucion(estado):
    """
    Verifica si un estado representa una solución válida al problema
    de las 8 reinas.

    Parámetros:
        estado (list): lista de 8 enteros, donde el índice representa
                       la columna y el valor representa la fila.

    Retorna:
        bool: True si es una solución válida, False en caso contrario.
    """
    if not isinstance(estado, list) or len(estado) != 8:
        return False

    # Verificar que todas las filas sean enteros entre 0 y 7
    for fila in estado:
        if not isinstance(fila, int) or fila < 0 or fila > 7:
            return False

    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            # Misma fila
            if r1 == r2:
                return False

            # Misma diagonal
            if abs(r1 - r2) == abs(c1 - c2):
                return False

    return True

print(es_solucion([0, 4, 7, 5, 2, 6, 1, 3]))  # True
print(es_solucion([0, 1, 2, 3, 4, 5, 6, 7]))  # False


def heuristica(estado):
    """
    Calcula el número de pares de reinas que se atacan entre sí.
    Un estado solución tiene heurística 0.
    """
    conflictos = 0
    n = 8

    for c1 in range(n):
        for c2 in range(c1 + 1, n):
            r1 = estado[c1]
            r2 = estado[c2]

            if r1 == r2 or abs(r1 - r2) == abs(c1 - c2):
                conflictos += 1

    return conflictos

True
False


## Hill Climbing

In [3]:
import random
import time

def estado_aleatorio(n=8):
    return [random.randint(0, n - 1) for _ in range(n)]

def vecinos(estado):
    n = len(estado)
    for col in range(n):
        fila_actual = estado[col]
        for nueva_fila in range(n):
            if nueva_fila != fila_actual:
                vecino = estado.copy()
                vecino[col] = nueva_fila
                yield vecino

def hill_climbing(estado_inicial):
    """
    Hill Climbing (steepest-ascent):
    - Evalúa todos los vecinos.
    - Se mueve al mejor vecino si mejora estrictamente la heurística.
    - Se detiene en mínimo local/meseta.
    """
    actual = estado_inicial.copy()
    h_actual = heuristica(actual)
    pasos = 0

    while True:
        mejor_h = h_actual
        mejores_vecinos = []

        for v in vecinos(actual):
            h_v = heuristica(v)
            if h_v < mejor_h:
                mejor_h = h_v
                mejores_vecinos = [v]
            elif h_v == mejor_h and h_v < h_actual:
                mejores_vecinos.append(v)

        if not mejores_vecinos:  # No hay mejora estricta
            break

        actual = random.choice(mejores_vecinos)  # desempate aleatorio
        h_actual = mejor_h
        pasos += 1

        if h_actual == 0:
            break

    return actual, pasos, h_actual, es_solucion(actual)

# Experimento requerido: 1000 ejecuciones con estados iniciales aleatorios
N_EJECUCIONES = 1000

exitos = 0
largos = []
heuristicas_finales = []
tiempos = []

for _ in range(N_EJECUCIONES):
    inicio = estado_aleatorio()
    t0 = time.perf_counter()
    estado_final, pasos, h_final, exito = hill_climbing(inicio)
    t1 = time.perf_counter()

    exitos += int(exito)
    largos.append(pasos)
    heuristicas_finales.append(h_final)
    tiempos.append(t1 - t0)

print("=== Resultados Hill Climbing (1000 ejecuciones) ===")
print(f"Porcentaje de éxito: {100 * exitos / N_EJECUCIONES:.2f}%")
print(f"Promedio largo de episodio: {sum(largos) / N_EJECUCIONES:.2f}")
print(f"Promedio heurística final: {sum(heuristicas_finales) / N_EJECUCIONES:.2f}")
print(f"Tiempo promedio de ejecución: {sum(tiempos) / N_EJECUCIONES:.6f} s")

=== Resultados Hill Climbing (1000 ejecuciones) ===
Porcentaje de éxito: 13.70%
Promedio largo de episodio: 3.22
Promedio heurística final: 1.30
Tiempo promedio de ejecución: 0.000420 s


## Hill Climbing (Stochastic/ First-choice/ Random-restart)

## Beam Search